# Evaluate the three Fed-MOPSD Client Students

Each saved Client Student is evaluated only on its own domain: Math on AIME25/AIME26, IF on IFBench/IFEval, and Code on LiveCodeBench release_v6. Models are loaded sequentially on one visible GPU and released after each domain.

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "1800")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
print("HF mirror:", os.environ["HF_ENDPOINT"], flush=True)
print("Physical CUDA device exposed:", os.environ["CUDA_VISIBLE_DEVICES"], flush=True)


In [ ]:
# Papermill parameters
CLIENT_MODELS_ROOT = "/home/lurui/code/Federated_Learning/results/fed_mopsd_qwen3_4b_base/training"
OUTPUT_DIR = "results/fed_mopsd_qwen3_4b_base/client_student_evaluation"
NLTK_DATA_DIR = "~/nltk_data"
LCB_REPO_DIR = "LiveCodeBench"
SEED = 42
RESUME = True
LOCAL_FILES_ONLY = True

LIMIT_AIME25 = None
LIMIT_AIME26 = None
LIMIT_IFBENCH = None
LIMIT_IFEVAL = None
LIMIT_LCB = None

EVAL_MAX_SEQ_LENGTH = 32768
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0
MAX_CONSECUTIVE_ERRORS = 3

LCB_RELEASE_VERSION = "release_v6"
LCB_DATASET_REVISION = "d44be6b144381afa49392b1f0eb424a64a4d8a10"
LCB_REPO_REVISION = "28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24"
LCB_NUM_PROCESSES = 4
LCB_TIMEOUT_SECONDS = 6
RUN_LCB_EVALUATION = True


In [ ]:
import gc
import hashlib
import importlib.util
import json
import math
import random
import re
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# Unsloth must be imported before transformers.
from unsloth import FastLanguageModel
import numpy as np
import pandas as pd
import torch
from datasets import DownloadConfig, disable_progress_bars, load_dataset
from huggingface_hub import snapshot_download

disable_progress_bars()
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for model evaluation.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"Exactly one visible GPU is required, got {torch.cuda.device_count()}.")
torch.cuda.set_device(0)
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as exc:
    print(f"[WARN] Could not enable deterministic algorithms: {exc!r}", flush=True)

ROOT = Path(CLIENT_MODELS_ROOT).expanduser().resolve()
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DOWNLOAD_CONFIG = DownloadConfig(local_files_only=bool(LOCAL_FILES_ONLY))
print(f"Logical cuda:0 -> {torch.cuda.get_device_name(0)}; dtype={DTYPE}; load_in_4bit=False", flush=True)
print(f"Client model root: {ROOT}", flush=True)
print(f"Output directory: {OUTPUT_DIR}", flush=True)

def resolve_client_model(domain):
    client_dir = ROOT / f"client_{domain}"
    candidates = [
        client_dir,
        client_dir / "student_model",
        ROOT / "round_000" / f"client_{domain}" / "student_model",
    ]
    for candidate in candidates:
        if (candidate / "config.json").is_file() and (
            (candidate / "model.safetensors.index.json").is_file()
            or (candidate / "model.safetensors").is_file()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find a complete client_{domain} model. Checked: "
        + ", ".join(map(str, candidates))
    )

CLIENT_MODEL_DIRS = {domain: resolve_client_model(domain) for domain in ("math", "if", "code")}
print(json.dumps({k: str(v) for k, v in CLIENT_MODEL_DIRS.items()}, indent=2), flush=True)

def model_manifest(model_dir):
    files = []
    for pattern in ("config.json", "model.safetensors", "model-*.safetensors", "model.safetensors.index.json"):
        for path in sorted(model_dir.glob(pattern)):
            stat = path.stat()
            files.append({"name": path.name, "size": stat.st_size, "mtime_ns": stat.st_mtime_ns})
    return files

def make_run_fingerprint(domain, model_dir):
    payload = {
        "method": "fed-mopsd-client-domain-eval-v1",
        "domain": domain, "model_dir": str(model_dir), "manifest": model_manifest(model_dir),
        "max_seq_length": EVAL_MAX_SEQ_LENGTH, "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K,
        "min_p": MIN_P, "seed": SEED,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:16]

def domain_paths(domain):
    domain_dir = OUTPUT_DIR / domain
    raw_dir = domain_dir / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    return domain_dir, raw_dir

def limit_dataset(dataset, limit):
    if limit is None:
        return dataset
    return dataset.select(range(min(int(limit), len(dataset))))

def format_seconds(seconds):
    seconds = int(max(0, seconds))
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"

def print_progress(label, done, total, started_at):
    percent = 100.0 * done / total if total else 100.0
    print(f"[PROGRESS] {label}: {done}/{total} ({percent:.2f}%) elapsed={format_seconds(time.time() - started_at)}", flush=True)

def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def load_matching_records(path, run_fingerprint):
    latest = {}
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"[WARN] Ignoring truncated JSONL line {path}:{line_number}", flush=True)
                continue
            if record.get("run_fingerprint") == run_fingerprint:
                latest[str(record["sample_id"])] = record
    return latest

def split_thinking(generated_text):
    text = generated_text.strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False

def load_client_model(domain):
    model_dir = CLIENT_MODEL_DIRS[domain]
    run_fingerprint = make_run_fingerprint(domain, model_dir)
    print(f"[LOAD] client-{domain}: {model_dir}; fingerprint={run_fingerprint}", flush=True)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(model_dir), max_seq_length=EVAL_MAX_SEQ_LENGTH, dtype=DTYPE,
        load_in_4bit=False, device_map={"": 0}, local_files_only=True,
    )
    FastLanguageModel.for_inference(model)
    model.eval()
    model.generation_config.max_length = None
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model.config.pad_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.eos_token_id
    if not tokenizer.chat_template:
        raise RuntimeError(f"Saved client-{domain} tokenizer has no chat_template.")
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    eos_ids = [int(tokenizer.eos_token_id)]
    if isinstance(im_end, int) and im_end >= 0 and im_end not in eos_ids:
        eos_ids.append(im_end)
    assert not bool(getattr(model, "is_loaded_in_4bit", False))
    assert {str(p.device) for p in model.parameters()} == {"cuda:0"}
    print(f"[LOAD] client-{domain} CUDA allocation: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB", flush=True)
    return model, tokenizer, eos_ids, run_fingerprint

def cleanup_cuda(domain):
    # Call only after deleting the caller's model/tokenizer references.
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"[UNLOAD] client-{domain}; CUDA allocation={torch.cuda.memory_allocated(0) / 2**30:.2f} GiB", flush=True)

def stable_seed(sample_id):
    # Identical to FedAvg_mopsd.ipynb: the same benchmark sample receives the same
    # sampling seed for every model, making Client-vs-global comparisons controlled.
    digest = hashlib.sha256(str(sample_id).encode("utf-8")).digest()
    return (SEED + int.from_bytes(digest[:4], "big")) % (2**31 - 1)

def generate_one(model, tokenizer, eos_ids, run_fingerprint, user_prompt, sample_id, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to("cuda:0")
    attention_mask = encoded["attention_mask"].to("cuda:0")
    input_tokens = int(input_ids.shape[1])
    available = EVAL_MAX_SEQ_LENGTH - input_tokens
    if available < 1:
        raise ValueError(f"Prompt has {input_tokens} tokens and exceeds EVAL_MAX_SEQ_LENGTH.")
    effective_max_new_tokens = min(MAX_NEW_TOKENS, available)
    sample_seed = stable_seed(sample_id)
    random.seed(sample_seed); np.random.seed(sample_seed); torch.manual_seed(sample_seed); torch.cuda.manual_seed_all(sample_seed)
    started = time.time()
    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=effective_max_new_tokens,
            do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, min_p=MIN_P,
            eos_token_id=eos_ids, pad_token_id=tokenizer.eos_token_id, use_cache=True,
        )
    new_tokens = generated[0, input_tokens:]
    raw_generation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    thinking, final_answer, found_think_close = split_thinking(raw_generation)
    return {
        "sample_seed": sample_seed, "raw_generation": raw_generation, "thinking": thinking,
        "final_answer": final_answer, "found_think_close": found_think_close,
        "input_tokens": input_tokens, "output_tokens": int(new_tokens.numel()),
        "effective_max_new_tokens": effective_max_new_tokens,
        "generation_seconds": time.time() - started,
    }

def metric_row(benchmark, domain, metric, numerator, denominator):
    score = numerator / denominator if denominator else None
    return {"benchmark": benchmark, "domain": domain, "metric": metric, "score": score,
            "score_percent": 100 * score if score is not None else None,
            "numerator": numerator, "denominator": denominator, "status": "complete", "note": ""}

def write_domain_metrics(domain, model_dir, run_fingerprint, rows, extra=None):
    domain_dir, _ = domain_paths(domain)
    payload = {"domain": domain, "model_dir": str(model_dir), "run_fingerprint": run_fingerprint,
               "created_at_utc": datetime.now(timezone.utc).isoformat(), "metrics": rows}
    if extra:
        payload.update(extra)
    (domain_dir / "metrics.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    pd.DataFrame(rows).to_csv(domain_dir / "metrics.csv", index=False, encoding="utf-8-sig")
    return payload


## Math Client Student → AIME25 and AIME26

In [ ]:
math_domain_dir, math_raw_dir = domain_paths("math")
math_datasets = {
    "aime25": limit_dataset(load_dataset("math-ai/aime25", split="test", download_config=LOCAL_DOWNLOAD_CONFIG), LIMIT_AIME25),
    "aime26": limit_dataset(load_dataset("math-ai/aime26", split="test", download_config=LOCAL_DOWNLOAD_CONFIG), LIMIT_AIME26),
}
for name, dataset in math_datasets.items():
    assert {"problem", "answer"}.issubset(dataset.column_names)
print({name: len(ds) for name, ds in math_datasets.items()}, flush=True)

def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", text)]
    for start in reversed(starts):
        brace_start = text.find("{", start); depth = 0
        for index in range(brace_start, len(text)):
            if text[index] == "{": depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0: return text[brace_start + 1:index]
    return None

def extract_math_answer(text):
    boxed = extract_last_boxed(text)
    if boxed is not None: return boxed
    final_matches = re.findall(r"(?i)final\s+answer\s*(?:is|:|=)?\s*([^\n]+)", text)
    if final_matches: return final_matches[-1].strip()
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", text.replace(",", ""))
    return integers[-1] if integers else ""

def normalize_math_answer(value):
    value = str(value).strip().replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value): value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value): value = str(int(value))
    return value

math_model, math_tokenizer, math_eos_ids, math_run_fp = load_client_model("math")
math_records = {}
for name, dataset in math_datasets.items():
    output_path = math_raw_dir / f"{name}.jsonl"
    prior = load_matching_records(output_path, math_run_fp) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(row.get("id", index)) for index, row in enumerate(dataset)}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] client-math/{name}: all {len(dataset)} completed", flush=True)
        math_records[name] = prior
        continue
    started = time.time(); done = 0; consecutive_errors = 0
    if completed: print(f"[RESUME] client-math/{name}: {len(completed)} completed", flush=True)
    for index, example in enumerate(dataset):
        sample_id = str(example.get("id", index))
        if sample_id in completed:
            done += 1; continue
        base = {"run_fingerprint": math_run_fp, "benchmark": name, "sample_id": sample_id,
                "problem": example["problem"], "gold_answer": str(example["answer"])}
        try:
            prompt = str(example["problem"]).strip() + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
            result = generate_one(math_model, math_tokenizer, math_eos_ids, math_run_fp, prompt, f"{name}:{sample_id}")
            predicted = extract_math_answer(result["final_answer"])
            record = {**base, **result, "status": "ok", "predicted_answer": predicted,
                      "correct": normalize_math_answer(predicted) == normalize_math_answer(example["answer"])}
            consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "correct": False}
            consecutive_errors += 1; print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True); torch.cuda.empty_cache()
        append_jsonl(output_path, record); prior[sample_id] = record; done += 1
        print_progress(f"client-math/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS: raise RuntimeError(f"Aborting {name}: {record.get('error')}")
    math_records[name] = load_matching_records(output_path, math_run_fp)

math_metric_rows = []
for name, dataset in math_datasets.items():
    selected_ids = [str(row.get("id", index)) for index, row in enumerate(dataset)]
    records = [math_records[name].get(sample_id, {}) for sample_id in selected_ids]
    math_metric_rows.append(metric_row(name.upper(), "Math", "exact_match", sum(bool(r.get("correct")) for r in records), len(records)))
write_domain_metrics("math", CLIENT_MODEL_DIRS["math"], math_run_fp, math_metric_rows)
display(pd.DataFrame(math_metric_rows))
del math_model, math_tokenizer
cleanup_cuda("math")


## Instruction-Following Client Student → IFBench and IFEval

In [ ]:
# Fixed official NLTK resources used by IFBench verifiers.
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
import nltk
try:
    import nltk.pathsec
    nltk.pathsec.ALLOW_PROXIED_FETCH = True
except (ImportError, AttributeError):
    pass
NLTK_DATA_DIR = Path(NLTK_DATA_DIR).expanduser().resolve()
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ["NLTK_DATA"] = str(NLTK_DATA_DIR)
if str(NLTK_DATA_DIR) not in nltk.data.path: nltk.data.path.insert(0, str(NLTK_DATA_DIR))
for package_name, resource_path in {"punkt_tab": "tokenizers/punkt_tab", "stopwords": "corpora/stopwords", "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng"}.items():
    try:
        nltk.data.find(resource_path); print(f"[NLTK] Found {package_name}", flush=True)
    except LookupError:
        print(f"[NLTK] Downloading {package_name}", flush=True)
        nltk.download(package_name, download_dir=str(NLTK_DATA_DIR), quiet=True, raise_on_error=True)
        nltk.data.find(resource_path)
from ifbench import instructions_registry

if_domain_dir, if_raw_dir = domain_paths("if")
if_datasets = {
    "ifbench": limit_dataset(load_dataset("allenai/IFBench_test", split="train", download_config=LOCAL_DOWNLOAD_CONFIG), LIMIT_IFBENCH),
    "ifeval": limit_dataset(load_dataset("google/IFEval", split="train", download_config=LOCAL_DOWNLOAD_CONFIG), LIMIT_IFEVAL),
}
for name, dataset in if_datasets.items():
    assert {"key", "prompt", "instruction_id_list", "kwargs"}.issubset(dataset.column_names)
print({name: len(ds) for name, ds in if_datasets.items()}, flush=True)

def response_variants(response):
    lines = response.split("\n")
    remove_first = "\n".join(lines[1:]).strip(); remove_last = "\n".join(lines[:-1]).strip(); remove_both = "\n".join(lines[1:-1]).strip()
    return [response, response.replace("*", ""), remove_first, remove_last, remove_both,
            remove_first.replace("*", ""), remove_last.replace("*", ""), remove_both.replace("*", "")]

def check_instruction(instruction_id, kwargs, prompt, candidates):
    instruction_cls = instructions_registry.INSTRUCTION_DICT[instruction_id]
    instruction = instruction_cls(instruction_id)
    instruction.build_description(**{k: v for k, v in dict(kwargs).items() if v is not None})
    args = instruction.get_instruction_args()
    if args and "prompt" in args: instruction.build_description(prompt=prompt)
    return any(candidate.strip() and instruction.check_following(candidate) for candidate in candidates)

def score_instruction_response(example, response):
    strict, loose = [], []
    for instruction_id, kwargs in zip(example["instruction_id_list"], example["kwargs"]):
        strict.append(check_instruction(instruction_id, kwargs, example["prompt"], [response]))
        loose.append(check_instruction(instruction_id, kwargs, example["prompt"], response_variants(response)))
    return strict, loose

if_model, if_tokenizer, if_eos_ids, if_run_fp = load_client_model("if")
if_records = {}
for name, dataset in if_datasets.items():
    output_path = if_raw_dir / f"{name}.jsonl"
    prior = load_matching_records(output_path, if_run_fp) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(row.get("key", index)) for index, row in enumerate(dataset)}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] client-if/{name}: all {len(dataset)} completed", flush=True); if_records[name] = prior; continue
    started = time.time(); done = 0; consecutive_errors = 0
    if completed: print(f"[RESUME] client-if/{name}: {len(completed)} completed", flush=True)
    for index, example in enumerate(dataset):
        sample_id = str(example.get("key", index))
        if sample_id in completed: done += 1; continue
        base = {"run_fingerprint": if_run_fp, "benchmark": name, "sample_id": sample_id,
                "prompt": example["prompt"], "instruction_id_list": example["instruction_id_list"], "kwargs": example["kwargs"]}
        try:
            result = generate_one(if_model, if_tokenizer, if_eos_ids, if_run_fp, example["prompt"], f"{name}:{sample_id}")
            strict, loose = score_instruction_response(example, result["final_answer"])
            record = {**base, **result, "status": "ok", "strict_instruction_results": strict,
                      "loose_instruction_results": loose, "strict_prompt_pass": all(strict), "loose_prompt_pass": all(loose)}
            consecutive_errors = 0
        except Exception as exc:
            failed = [False] * len(example["instruction_id_list"]); consecutive_errors += 1
            record = {**base, "status": "error", "error": repr(exc), "strict_instruction_results": failed,
                      "loose_instruction_results": failed, "strict_prompt_pass": False, "loose_prompt_pass": False}
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True); torch.cuda.empty_cache()
        append_jsonl(output_path, record); prior[sample_id] = record; done += 1
        print_progress(f"client-if/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS: raise RuntimeError(f"Aborting {name}: {record.get('error')}")
    if_records[name] = load_matching_records(output_path, if_run_fp)

if_metric_rows = []
for name, dataset in if_datasets.items():
    ids = [str(row.get("key", index)) for index, row in enumerate(dataset)]
    records = [if_records[name].get(sample_id, {}) for sample_id in ids]
    strict_lists = [r.get("strict_instruction_results", []) for r in records]
    loose_lists = [r.get("loose_instruction_results", []) for r in records]
    instruction_total = sum(len(values) for values in strict_lists)
    if_metric_rows.extend([
        metric_row(name.upper(), "Instruction Following", "prompt_level_strict", sum(bool(v) and all(v) for v in strict_lists), len(records)),
        metric_row(name.upper(), "Instruction Following", "instruction_level_strict", sum(sum(v) for v in strict_lists), instruction_total),
        metric_row(name.upper(), "Instruction Following", "prompt_level_loose", sum(bool(v) and all(v) for v in loose_lists), len(records)),
        metric_row(name.upper(), "Instruction Following", "instruction_level_loose", sum(sum(v) for v in loose_lists), instruction_total),
    ])
write_domain_metrics("if", CLIENT_MODEL_DIRS["if"], if_run_fp, if_metric_rows)
display(pd.DataFrame(if_metric_rows))
del if_model, if_tokenizer
cleanup_cuda("if")


## Code Client Student → LiveCodeBench release_v6

In [ ]:
code_domain_dir, code_raw_dir = domain_paths("code")
lcb_repo_path = Path(LCB_REPO_DIR).expanduser().resolve()
if not (lcb_repo_path / "lcb_runner").is_dir():
    raise FileNotFoundError(f"LiveCodeBench source not found: {lcb_repo_path / 'lcb_runner'}")
actual_lcb_revision = subprocess.check_output(["git", "-C", str(lcb_repo_path), "rev-parse", "HEAD"], text=True).strip()
if actual_lcb_revision != LCB_REPO_REVISION:
    raise RuntimeError(f"LiveCodeBench revision mismatch: expected {LCB_REPO_REVISION}, got {actual_lcb_revision}")
if str(lcb_repo_path) not in sys.path: sys.path.insert(0, str(lcb_repo_path))
from lcb_runner.benchmarks.code_generation import CodeGenerationProblem

lcb_snapshot = Path(snapshot_download(
    repo_id="livecodebench/code_generation_lite", repo_type="dataset", revision=LCB_DATASET_REVISION,
    allow_patterns=[f"{LCB_RELEASE_VERSION}/*.parquet"], local_files_only=bool(LOCAL_FILES_ONLY),
)).resolve()
lcb_parquet_files = [lcb_snapshot / LCB_RELEASE_VERSION / f"test-{index:05d}-of-00009.parquet" for index in range(9)]
missing_lcb = [str(path) for path in lcb_parquet_files if not path.is_file()]
if missing_lcb: raise FileNotFoundError("Missing cached LiveCodeBench shards: " + ", ".join(missing_lcb))
print(f"[CACHE] Loading 9 local LiveCodeBench shards from {lcb_snapshot / LCB_RELEASE_VERSION}", flush=True)
raw_lcb = load_dataset("parquet", data_files={"test": [str(p) for p in lcb_parquet_files]}, split="test", download_config=LOCAL_DOWNLOAD_CONFIG)
lcb_dataset = sorted([CodeGenerationProblem(**row) for row in raw_lcb], key=lambda p: str(p.question_id))
if LIMIT_LCB is not None: lcb_dataset = lcb_dataset[:min(int(LIMIT_LCB), len(lcb_dataset))]
print(f"LiveCodeBench rows: {len(lcb_dataset)}", flush=True)

notebook_cwd = Path.cwd()
prompt_module_path = lcb_repo_path / "lcb_runner" / "prompts" / "code_generation.py"
prompt_spec = importlib.util.spec_from_file_location("_pinned_lcb_code_generation_prompt", prompt_module_path)
if prompt_spec is None or prompt_spec.loader is None: raise ImportError(f"Unable to load {prompt_module_path}")
prompt_module = importlib.util.module_from_spec(prompt_spec)
try:
    os.chdir(lcb_repo_path); prompt_spec.loader.exec_module(prompt_module)
finally:
    os.chdir(notebook_cwd)
PromptConstants = prompt_module.PromptConstants
get_generic_question_template_answer = prompt_module.get_generic_question_template_answer
from lcb_runner.lm_styles import LMStyle
from lcb_runner.utils.extraction_utils import extract_code

code_model, code_tokenizer, code_eos_ids, code_run_fp = load_client_model("code")
lcb_run_fp = hashlib.sha256(json.dumps({"run": code_run_fp, "repo": LCB_REPO_REVISION, "dataset": LCB_DATASET_REVISION,
                                             "release": LCB_RELEASE_VERSION, "prompt": "official-generic-chat-v1"},
                                            sort_keys=True).encode("utf-8")).hexdigest()[:16]
lcb_output_path = code_raw_dir / "livecodebench.jsonl"
prior = load_matching_records(lcb_output_path, code_run_fp) if RESUME else {}
prior = {k: v for k, v in prior.items() if v.get("lcb_run_fingerprint") == lcb_run_fp}
completed = {key for key, value in prior.items() if value.get("status") == "ok"}
expected_ids = {str(problem.question_id) for problem in lcb_dataset}
if RESUME and expected_ids.issubset(completed):
    print(f"[SKIP] client-code/livecodebench: all {len(lcb_dataset)} completed", flush=True)
else:
    started = time.time(); done = 0; consecutive_errors = 0
    if completed: print(f"[RESUME] client-code/livecodebench: {len(completed)} completed", flush=True)
    for problem in lcb_dataset:
        sample_id = str(problem.question_id)
        if sample_id in completed: done += 1; continue
        base = {"run_fingerprint": code_run_fp, "lcb_run_fingerprint": lcb_run_fp, "benchmark": "livecodebench",
                "sample_id": sample_id, "question_title": problem.question_title, "question_content": problem.question_content,
                "platform": problem.platform.value, "contest_id": problem.contest_id,
                "contest_date": problem.contest_date.isoformat(), "difficulty": problem.difficulty.value, "starter_code": problem.starter_code}
        try:
            result = generate_one(code_model, code_tokenizer, code_eos_ids, code_run_fp,
                                  get_generic_question_template_answer(problem), f"livecodebench:{sample_id}",
                                  system_prompt=PromptConstants.SYSTEM_MESSAGE_GENERIC)
            extracted_code = extract_code(result["final_answer"], LMStyle.CodeQwenInstruct)
            record = {**base, **result, "status": "ok", "extracted_code": extracted_code}; consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "extracted_code": ""}; consecutive_errors += 1
            print(f"[ERROR] livecodebench:{sample_id}: {exc!r}", flush=True); torch.cuda.empty_cache()
        append_jsonl(lcb_output_path, record); prior[sample_id] = record; done += 1
        print_progress("client-code/livecodebench", done, len(lcb_dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS: raise RuntimeError(f"Aborting LiveCodeBench: {record.get('error')}")
prior = load_matching_records(lcb_output_path, code_run_fp)
lcb_records = {k: v for k, v in prior.items() if v.get("lcb_run_fingerprint") == lcb_run_fp}
del code_model, code_tokenizer
cleanup_cuda("code")

lcb_eval_path = code_domain_dir / "livecodebench_eval.json"
lcb_status_path = code_domain_dir / "livecodebench_eval_status.json"
expected_id_list = [str(problem.question_id) for problem in lcb_dataset]
missing_ids = [sample_id for sample_id in expected_id_list if lcb_records.get(sample_id, {}).get("status") != "ok"]
if missing_ids: raise RuntimeError(f"LiveCodeBench has {len(missing_ids)} missing generations; first: {missing_ids[:5]}")
expected_ids_hash = hashlib.sha256(json.dumps(expected_id_list).encode("utf-8")).hexdigest()
lcb_eval_fp = hashlib.sha256(json.dumps({"lcb_run": lcb_run_fp, "ids": expected_ids_hash, "timeout": LCB_TIMEOUT_SECONDS,
                                         "repo": LCB_REPO_REVISION}, sort_keys=True).encode("utf-8")).hexdigest()[:16]
previous_status = {}
if RESUME and lcb_status_path.is_file() and lcb_eval_path.is_file():
    try: previous_status = json.loads(lcb_status_path.read_text(encoding="utf-8"))
    except Exception: previous_status = {}
if previous_status.get("status") == "complete" and previous_status.get("eval_fingerprint") == lcb_eval_fp:
    lcb_evaluation = previous_status
    print(f"[SKIP] LiveCodeBench checker complete: {lcb_evaluation['correct']}/{lcb_evaluation['total']}", flush=True)
else:
    class LineProgress:
        def __init__(self, *args, total=None, **kwargs): self.total = int(total or 0); self.done = 0; self.started = time.time()
        def __enter__(self): return self
        def __exit__(self, exc_type, exc, traceback): return False
        def update(self, amount=1): self.done += int(amount); print_progress("client-code/livecodebench-check", self.done, self.total, self.started)
    import lcb_runner.evaluation.compute_code_generation_metrics as lcb_metric_module
    from lcb_runner.evaluation.pass_k_utils import extract_instance_results
    lcb_metric_module.tqdm = LineProgress
    samples_list = [problem.get_evaluation_sample() for problem in lcb_dataset]
    generations_list = [[lcb_records[str(problem.question_id)]["extracted_code"]] for problem in lcb_dataset]
    official_metrics, official_results, official_metadata = lcb_metric_module.codegen_metrics(
        samples_list, generations_list, k_list=[1], num_process_evaluate=int(LCB_NUM_PROCESSES),
        timeout=int(LCB_TIMEOUT_SECONDS), debug=False)
    instance_grades = extract_instance_results(official_results)
    graded = [bool(grades[0]) for grades in instance_grades]
    correct = sum(graded); total = len(graded); pass_at_1 = float(official_metrics["pass@1"])
    assert math.isclose(pass_at_1, correct / total, rel_tol=0.0, abs_tol=1e-12)
    eval_payload = {"release_version": LCB_RELEASE_VERSION, "repository_revision": LCB_REPO_REVISION,
                    "dataset_revision": LCB_DATASET_REVISION, "lcb_run_fingerprint": lcb_run_fp,
                    "eval_fingerprint": lcb_eval_fp, "official_metrics": official_metrics,
                    "correct": correct, "total": total, "pass@1": pass_at_1,
                    "per_problem": [{"question_id": expected_id_list[i], "passed": graded[i],
                                      "test_results": official_results[i][0], "metadata": official_metadata[i][0]} for i in range(total)]}
    lcb_eval_path.write_text(json.dumps(eval_payload, ensure_ascii=False, indent=2), encoding="utf-8")
    lcb_evaluation = {"status": "complete", "correct": correct, "total": total, "pass@1": pass_at_1,
                      "eval_fingerprint": lcb_eval_fp, "result_path": str(lcb_eval_path)}
    lcb_status_path.write_text(json.dumps(lcb_evaluation, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[RESULT] LiveCodeBench pass@1: {correct}/{total} = {100 * pass_at_1:.2f}%", flush=True)
code_metric_rows = [metric_row("LiveCodeBench", "Code", "pass@1", lcb_evaluation["correct"], lcb_evaluation["total"])]
write_domain_metrics("code", CLIENT_MODEL_DIRS["code"], code_run_fp, code_metric_rows, {"livecodebench_evaluation": lcb_evaluation})
display(pd.DataFrame(code_metric_rows))


## Combined metrics and bar chart

In [ ]:
all_metrics = []
domain_payloads = {}
for domain in ("math", "if", "code"):
    metrics_path = OUTPUT_DIR / domain / "metrics.json"
    if not metrics_path.is_file(): raise FileNotFoundError(metrics_path)
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    domain_payloads[domain] = payload
    all_metrics.extend(payload["metrics"])
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(OUTPUT_DIR / "client_students_metrics.csv", index=False, encoding="utf-8-sig")
summary = {"created_at_utc": datetime.now(timezone.utc).isoformat(), "evaluation_scope": {"math": ["AIME25", "AIME26"],
           "if": ["IFBENCH", "IFEVAL"], "code": ["LiveCodeBench"]}, "domains": domain_payloads, "metrics": all_metrics}
(OUTPUT_DIR / "client_students_metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
display(metrics_df)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
primary_specs = [("AIME25", "exact_match", "AIME25\nExact match"), ("AIME26", "exact_match", "AIME26\nExact match"),
                 ("IFBENCH", "prompt_level_loose", "IFBench\nPrompt loose"), ("IFEVAL", "prompt_level_loose", "IFEval\nPrompt loose"),
                 ("LiveCodeBench", "pass@1", "LiveCodeBench\nPass@1")]
labels, values = [], []
for benchmark, metric, label in primary_specs:
    row = metrics_df[(metrics_df["benchmark"] == benchmark) & (metrics_df["metric"] == metric)]
    if not row.empty: labels.append(label); values.append(float(row.iloc[0]["score_percent"]))
colors = ["#4C78A8", "#4C78A8", "#F58518", "#F58518", "#54A24B"][:len(labels)]
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="#333333", linewidth=0.7)
for bar, value in zip(bars, values): ax.text(bar.get_x() + bar.get_width()/2, value + 1.2, f"{value:.2f}%", ha="center")
ax.set_xticks(range(len(labels)), labels); ax.set_ylabel("Score (%)"); ax.set_ylim(0, 108)
ax.set_title("Fed-MOPSD Domain Client Student Evaluation"); ax.grid(axis="y", linestyle="--", alpha=0.3)
fig.tight_layout(); figure_path = OUTPUT_DIR / "client_students_scores.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight"); plt.show()
print(f"Metrics JSON: {OUTPUT_DIR / 'client_students_metrics.json'}", flush=True)
print(f"Metrics CSV:  {OUTPUT_DIR / 'client_students_metrics.csv'}", flush=True)
print(f"Bar chart:    {figure_path}", flush=True)
gc.collect(); torch.cuda.empty_cache()
print("[DONE] All three domain Client Students were evaluated on their corresponding benchmarks.", flush=True)
